In [ ]:
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
from pathlib import Path
from contextlib import ExitStack
import hashlib
import json
import os
import shutil
import tempfile
import warnings

RESULT_DIR = RESULTS
RIDGE_PRED_DIR = RESULT_DIR / 'ridge_baseline_alpha_extended'   # selected alpha = 1e-8
RIDGE_TRUTH_DIR = RESULT_DIR / 'ridge_baseline'
FIGURE_DIR = FIGURES
WORK_DIR = RESULTS / 'FigS21_extreme_baselines'
RAW_DIR = DATA
THRESHOLD_CSV = RESULTS / 'extreme_ETS_radar' / 'training_extreme_thresholds.csv'
FIG_DPI = 800
BLOCK_SIZE = 32
SHOW_FIGURE = True
FILE_OVERRIDES = {}       # ('B'/'Ridge', lead, 'u'/'v'/'w', 'testy'/'predicty') -> Path
VARIABLE_OVERRIDES = {}   # 同样的键 -> 精确变量名
RAW_FILE_OVERRIDES = {}   # ('w',850) -> 单层原始文件，若目录结构不变可留空

LEADS = tuple(range(1, 7))
GROUPS = ('B', 'Ridge', 'P')
KINDS = ('testy', 'predicty')
COMPONENTS = ('u', 'v', 'w')
WINDS = ('horizontal', 'vertical')
QUANTILES = (0.90, 0.95, 0.99)
TRAIN_SUPPORT = ('2000-01-01T12', '2016-10-19T10', 147263)
TEST_SUPPORT = ('2019-03-15T08', '2023-12-31T23', 42064)
EXPECTED_NATIVE_COUNT = {1: 42071, 2: 42071, 3: 42070, 4: 42070, 5: 42070, 6: 42070}
EXPECTED_GRID = ((24.,36.,49), (108.,126.,73))
RIDGE_ALPHA = 1e-8
W_TO_OMEGA = {'B': -10., 'P': -10., 'Ridge': 1.}
TIME_ROUND_TOL_NS = 1000
ALIASES = {'time': ('time','valid_time','times','date','datetime'),
           'level': ('level','levels','pressure_level','isobaricInhPa','isobaricInPa','plev'),
           'latitude': ('latitude','lat'), 'longitude': ('longitude','lon')}
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(WORK_DIR / 'matplotlib_config')
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, Normalize
from matplotlib.ticker import MaxNLocator
from PIL import Image
from IPython.display import display

LEVELS = np.array([850., 925., 1000.])
CONFIGS = [(g,h) for g in GROUPS for h in LEADS]
MODEL_NAMES = ('DA-TransUNet', 'Ridge', 'Persistence')
PAIR_INDEX = ((0,2),(0,1),(1,2))
PAIR_NAMES = ('DA - Persistence','DA - Ridge','Ridge - Persistence')
RAW_START = '2000-01-01T06'
RAW_END = '2023-12-31T23'
RAW_LAT_RANGE, RAW_LON_RANGE = EXPECTED_GRID[0][:2], EXPECTED_GRID[1][:2]
if FIG_DPI <= 600 or BLOCK_SIZE < 1:
    raise ValueError('FIG_DPI 必须大于600，BLOCK_SIZE须为正。')


In [ ]:
def rounded_hour_times(values, source):
    raw = np.asarray(values)
    if not np.issubdtype(raw.dtype, np.datetime64):
        raise TypeError(f'{source}: time须可解码为datetime64，不能将非标准日历静默转为标准日历。')
    raw = raw.astype('datetime64[ns]')
    if np.isnat(raw).any() or raw.size == 0:
        raise ValueError(f'{source}: 空时间轴或NaT。')
    ns = raw.astype(np.int64)
    if np.any(np.diff(ns) <= 0):
        raise ValueError(f'{source}: 原时间轴重复或倒序。')
    hour = 3_600_000_000_000
    quotient, remainder = np.divmod(ns, hour)
    rounded_ns = (quotient + (remainder >= hour // 2)) * hour
    delta = rounded_ns - ns
    if np.max(np.abs(delta)) > TIME_ROUND_TOL_NS:
        raise ValueError(f'{source}: 非整点偏差{np.max(np.abs(delta))} ns超过允许的编码噪声。')
    if np.any(np.diff(rounded_ns) <= 0):
        raise ValueError(f'{source}: 整点编码修正后出现重复或倒序。')
    changes = [dict(index=int(i), original=str(raw[i]), rounded=str(np.datetime64(int(rounded_ns[i]), 'ns')),
                    correction_ns=int(delta[i])) for i in np.flatnonzero(delta)]
    return rounded_ns.astype('datetime64[ns]'), changes


def canonical_field(ds, variable, source):
    if variable in ds.data_vars:
        da = ds[variable]
    else:
        candidates = []
        for name, value in ds.data_vars.items():
            if not np.issubdtype(value.dtype, np.number):
                continue
            if all(any(alias in value.dims for alias in aliases) for aliases in ALIASES.values()):
                candidates.append(name)
        if len(candidates) != 1:
            raise ValueError(f'{source}: 找不到{variable}且无法唯一确定四维风场，候选={candidates}。')
        da = ds[candidates[0]]
    rename = {}
    for standard, aliases in ALIASES.items():
        found = [a for a in aliases if a in da.dims]
        if len(found) != 1:
            raise ValueError(f'{source}: {standard}维度不存在或有歧义：{found}。')
        rename[found[0]] = standard
    da = da.rename({a: b for a, b in rename.items() if a != b})
    for dim in tuple(da.dims):
        if dim not in ALIASES:
            if da.sizes[dim] != 1:
                raise ValueError(f'{source}: 不能自动丢弃额外非单例维度{dim}。')
            da = da.isel({dim: 0}, drop=True)
    for dim in ALIASES:
        if dim not in da.coords or da[dim].ndim != 1:
            raise ValueError(f'{source}: {dim}必须有一维坐标。')
    lev = np.asarray(da.level.values, dtype=np.float64)
    units = str(da.level.attrs.get('units', '')).strip().lower().replace(' ', '')
    if units in ('pa', 'pascal', 'pascals') or (not units and np.all(lev > 2000)):
        lev = lev / 100.0
    elif units not in ('', 'hpa', 'hectopascal', 'hectopascals', 'mb', 'millibar', 'millibars'):
        raise ValueError(f'{source}: 未识别的气压层单位{units!r}。')
    if len(np.unique(lev)) != len(lev):
        raise ValueError(f'{source}: 气压层坐标重复。')
    indices = []
    for wanted in LEVELS:
        hit = np.flatnonzero(np.isclose(lev, wanted, rtol=0, atol=1e-8))
        if len(hit) != 1:
            raise ValueError(f'{source}: 缺少或重复气压层{wanted} hPa。')
        indices.append(int(hit[0]))
    da = da.assign_coords(level=lev).isel(level=indices)
    for dim in ('latitude', 'longitude'):
        c = np.asarray(da[dim].values, dtype=np.float64)
        if not np.isfinite(c).all() or len(np.unique(c)) != len(c):
            raise ValueError(f'{source}: {dim}坐标含非有限值或重复。')
        da = da.isel({dim: np.argsort(c)}).assign_coords({dim: np.sort(c)})
    da = da.transpose('time', 'level', 'latitude', 'longitude')
    return da


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)


def file_stat(path):
    stat = Path(path).stat()
    return dict(path=str(Path(path).resolve()), size=stat.st_size, mtime_ns=stat.st_mtime_ns)


def time_selection(indices):
    if len(indices) and (len(indices) == 1 or np.all(np.diff(indices) == 1)):
        return slice(int(indices[0]), int(indices[-1]) + 1)
    return indices


def nc_path(group, lead, component, kind):
    if group == 'P':
        return nc_path('B', lead, component, 'testy')
    key = (group, int(lead), component, kind)
    default = ((RIDGE_PRED_DIR if kind == 'predicty' else RIDGE_TRUTH_DIR) / f'Ridge_{lead}hour_{kind}.nc' if group == 'Ridge' else
               RESULT_DIR / f'da_transunet_{kind}_{component}_{lead}_24year_changjiang_residual.nc')
    return Path(FILE_OVERRIDES.get(key, default))


def variable_name(ds, group, lead, component, kind, path):
    source_group = 'B' if group == 'P' else group
    source_kind = 'testy' if group == 'P' else kind
    key = (source_group, int(lead), component, source_kind)
    if key in VARIABLE_OVERRIDES:
        name = VARIABLE_OVERRIDES[key]
        if name not in ds.data_vars:
            raise ValueError(f'{path}: 指定变量{name!r}不存在，实际为{list(ds.data_vars)}。')
        return name
    candidates = [f'{source_kind}_{component}', component]
    if component == 'w':
        candidates += [f'{source_kind}_omega', 'omega']
    for name in candidates:
        if name in ds.data_vars:
            return name
    hits = [name for name, da in ds.data_vars.items()
            if np.issubdtype(da.dtype, np.number)
            and all(any(a in da.dims for a in aliases) for aliases in ALIASES.values())]
    if len(hits) == 1 and source_group == 'B':
        return hits[0]
    raise ValueError(f'{path}: 无法识别{source_kind}/{component}，实际变量={list(ds.data_vars)}；'
                     '请在VARIABLE_OVERRIDES中填写精确映射。')


def scan_inputs():
    records, audit = {}, []
    common, latitude, longitude = None, None, None
    missing = sorted({str(nc_path(g, h, c, k)) for g, h in CONFIGS
                      for c in COMPONENTS for k in KINDS
                      if not nc_path(g, h, c, k).is_file()})
    if missing:
        raise FileNotFoundError('缺少以下文件，请检查目录或FILE_OVERRIDES：\n' + '\n'.join(missing))
    for group, lead in CONFIGS:
        for component in COMPONENTS:
            for kind in KINDS:
                key = (group, lead, component, kind)
                path = nc_path(*key)
                with xr.open_dataset(path, cache=False) as source:
                    variable = variable_name(source, *key, path)
                    field = canonical_field(source, variable, path)
                    native, corrections = rounded_hour_times(field.time.values, path)
                    lat, lon = field.latitude.values.copy(), field.longitude.values.copy()
                    if latitude is None:
                        latitude, longitude = lat, lon
                    elif not (np.array_equal(latitude, lat) and np.array_equal(longitude, lon)):
                        raise ValueError(f'{path}: 网格不一致，不进行空间插值。')
                    source_units = str(field.attrs.get('units', 'not specified'))
                target = native + np.timedelta64(lead, 'h') if group == 'P' and kind == 'predicty' else native
                common = target if common is None else np.intersect1d(common, target, assume_unique=True)
                records[key] = dict(path=path, variable=variable, native_times=native, times=target, stat=file_stat(path))
                audit.append(dict(group=group, lead_hour=lead, component=component, kind=kind,
                                  variable=variable, count=len(native), native_start=str(native[0]),
                                  native_end=str(native[-1]), stored_units=source_units,
                                  conversion_to_physical=W_TO_OMEGA[group] if component == 'w' else 1.,
                                  n_time_round_corrections=len(corrections), **file_stat(path)))
    for h in LEADS:
        baseline = records[('B', h, 'u', 'testy')]['native_times']
        if EXPECTED_NATIVE_COUNT is not None and len(baseline) != EXPECTED_NATIVE_COUNT[h]:
            raise ValueError(f'{h}h原生样本数{len(baseline)}与验收记录{EXPECTED_NATIVE_COUNT[h]}不符。请核对文件。')
        for group in ('B', 'Ridge'):
            for c in COMPONENTS:
                for kind in KINDS:
                    if not np.array_equal(records[(group, h, c, kind)]['native_times'], baseline):
                        raise ValueError(f'{group}/{h}h/{c}/{kind}: 原生时间轴与该时效Fig7基准不一致。')
    if common is None or len(common) < 3:
        raise ValueError('共同评价时次不足3个。')
    if np.any(np.diff(common) != np.timedelta64(1, 'h')):
        raise ValueError('共同评价期存在非逐小时间断，请核对文件；不自动填补。')
    for entry, record in zip(audit, records.values()):
        idx = np.searchsorted(record['times'], common)
        if not np.array_equal(record['times'][idx], common):
            raise AssertionError('时间戳索引失败。')
        record['indices'] = idx
        excluded = np.setdiff1d(record['times'], common)
        entry.update(evaluation_count=len(common), excluded_count=len(excluded),
                     excluded_first=str(excluded[0]) if len(excluded) else None,
                     excluded_last=str(excluded[-1]) if len(excluded) else None)
    return records, common, latitude, longitude, audit


def assert_sources_unchanged():
    for record in records.values():
        if file_stat(record['path']) != record['stat']:
            raise RuntimeError(f"源文件发生变化：{record['path']}。请重新运行输入核对框。")


def exact_truth_check(a, b, label):
    if a.shape != b.shape or not np.array_equal(a, b, equal_nan=True):
        good = np.isfinite(a) & np.isfinite(b) if a.shape == b.shape else np.array(False)
        delta = float(np.max(np.abs(a[good] - b[good]))) if good.any() else None
        raise ValueError(f'{label}: Ridge与原模型物理单位真值不完全一致，最大有限差={delta}。'
                         '请核对源文件和w转换；不自动放宽容差或替换真值。')


def write_nc(ds, path):
    path = Path(path)
    temporary = path.with_name(path.stem + '.tmp.nc')
    ds.to_netcdf(temporary, engine='netcdf4',
                 encoding={k: {'zlib': True, 'complevel': 4} for k in ds.data_vars})
    with xr.open_dataset(temporary) as check:
        check.load()
        if dict(check.sizes) != dict(ds.sizes):
            raise RuntimeError('指标NC读回维度不一致。')
        for name in ds.data_vars:
            if not np.array_equal(check[name].values, ds[name].values, equal_nan=True):
                raise RuntimeError(f'指标NC读回不一致：{name}')
    os.replace(temporary, path)


def audit_ridge_headers():
    rows = []
    for h in LEADS:
        pred = nc_path('Ridge', h, 'u', 'predicty')
        truth = nc_path('Ridge', h, 'u', 'testy')
        if pred.resolve() == (RIDGE_TRUTH_DIR / f'Ridge_{h}hour_predicty.nc').resolve():
            raise ValueError(f'{h}h误用了旧alpha=1e-6预测：{pred}')
        with xr.open_dataset(pred, cache=False) as ds:
            found = {}
            for name in ('alpha', 'ridge_alpha', 'selected_alpha', 'regularization_alpha'):
                if name in ds.attrs:
                    value = float(ds.attrs[name])
                    if not np.isclose(value, RIDGE_ALPHA, rtol=1e-10, atol=0):
                        raise ValueError(f'{pred}: {name}={value}，不是要求的{RIDGE_ALPHA}。')
                    found[name] = value
            if 'downstream_w_scale' in ds.attrs and float(ds.attrs['downstream_w_scale']) != 1.:
                raise ValueError(f'{pred}: downstream_w_scale必须为1。')
        rows.append(dict(lead_hour=h, declared_alpha=RIDGE_ALPHA, predicty=str(pred), testy=str(truth),
                         alpha_fields_in_nc=found,
                         provenance='New prediction directory from the supplied final boundary report; '
                                    'numeric alpha metadata checked when present'))
    return rows


def raw_truth_path(component,level):
    prefix={'u':'U-component-of-wind','v':'V-component-of-wind','w':'Vertical-velocity'}[component]
    return Path(RAW_FILE_OVERRIDES.get((component,int(level)),RAW_DIR/f'{prefix}-{int(level)}hpa.nc'))


def canonical_raw_level(ds,component,level,path):
    if component not in ds: raise ValueError(f'{path}: 缺少原训练变量{component}。')
    a=ds[component]
    for dim in list(a.dims):
        if dim not in sum((ALIASES[k] for k in ('time','latitude','longitude')),()):
            if a.sizes[dim]!=1: raise ValueError(f'{path}: 单层源文件存在额外非单例维度{dim}。')
            if dim in ALIASES['level'] and dim in a.coords:
                v=float(a[dim].values[0]);units=str(a[dim].attrs.get('units','')).lower()
                if units=='pa' or v>2000: v/=100
                if not np.isclose(v,level): raise ValueError(f'{path}: 文件内气压层与文件名不一致。')
            a=a.isel({dim:0},drop=True)
    rename={}
    for name in ('time','latitude','longitude'):
        hits=[d for d in ALIASES[name] if d in a.dims]
        if len(hits)!=1 or hits[0] not in a.coords: raise ValueError(f'{path}: 缺少唯一{name}坐标。')
        rename[hits[0]]=name
    a=a.rename({k:v for k,v in rename.items() if k!=v})
    tt,_=rounded_hour_times(a.time.values,path)
    a=a.assign_coords(time=tt)
    for dim in ('latitude','longitude'):
        values=a[dim].values
        if not np.isfinite(values).all() or len(np.unique(values))!=len(values): raise ValueError(f'{path}: {dim}坐标无效。')
        a=a.sortby(dim)
    start=np.datetime64(RAW_START,'ns');end=np.datetime64(RAW_END,'ns')
    if start not in tt or end not in tt: raise ValueError(f'{path}: 原训练完整起止时间不齐，不能按缺失后的长度重新切分。')
    a=a.sel(time=slice(start,end),latitude=slice(RAW_LAT_RANGE[0],RAW_LAT_RANGE[1]),longitude=slice(RAW_LON_RANGE[0],RAW_LON_RANGE[1]))
    if min(a.sizes.values())==0: raise ValueError(f'{path}: 原训练区域选择后为空。')
    if np.any(np.diff(a.time.values)!=np.timedelta64(1,'h')): raise ValueError(f'{path}: 原训练时间范围内缺小时。')
    return a.transpose('time','latitude','longitude')


In [ ]:
def validate_train_table(table):
    required = {'wind','level_hpa','quantile','threshold','units','reference_split',
                'reference_start','reference_end','training_time_count'}
    if missing := required.difference(table.columns):
        raise ValueError(f'训练阈值表缺少来源或字段 {sorted(missing)}。可将 THRESHOLD_CSV 设为 None 从原场恢复。')
    if len(table) != len(WINDS)*len(LEVELS)*len(QUANTILES):
        raise ValueError('阈值表必须恰有18条记录。')
    values = np.empty((len(WINDS),len(LEVELS),len(QUANTILES)))
    start, stop, count = TRAIN_SUPPORT
    for wi, wind in enumerate(WINDS):
        for li, level in enumerate(LEVELS):
            for qi, q in enumerate(QUANTILES):
                row = table[(table.wind==wind) & (table.level_hpa==level) & np.isclose(table['quantile'],q)]
                if len(row)!=1: raise ValueError(f'阈值不唯一 {wind}/{level}/{q}')
                r=row.iloc[0]
                if (r.reference_split!='train' or pd.to_datetime(r.reference_start,utc=True)!=pd.to_datetime(start,utc=True)
                    or pd.to_datetime(r.reference_end,utc=True)!=pd.to_datetime(stop,utc=True)
                    or int(r.training_time_count)!=count or r.units!=('m/s' if wi==0 else 'Pa/s')):
                    raise ValueError(f'训练阈值来源或单位不符 {wind}/{level}/{q}')
                values[wi,li,qi]=float(r.threshold)
    if not np.all(np.isfinite(values) & (values>=0)) or np.any(np.diff(values,axis=-1)<0):
        raise ValueError('阈值无效或未随分位数递增。')
    return values


def obtain_thresholds():
    if THRESHOLD_CSV is not None and Path(THRESHOLD_CSV).is_file():
        table=pd.read_csv(THRESHOLD_CSV, float_precision='round_trip')
        return validate_train_table(table), str(THRESHOLD_CSV)
    start,stop,ntrain=TRAIN_SUPPORT
    expected=pd.date_range(start,stop,freq='h').to_numpy(dtype='datetime64[ns]')
    if len(expected)!=ntrain: raise ValueError('训练时次配置不一致。')
    rows=[]
    for wi,wind in enumerate(WINDS):
        for li,level in enumerate(LEVELS):
            comps=('u','v') if wi==0 else ('w',)
            paths=[raw_truth_path(c,level) for c in comps]
            if not all(p.is_file() for p in paths):
                raise FileNotFoundError(f'找不到原始风场：{paths}。可在 THRESHOLD_CSV 指向原实验的 training_extreme_thresholds.csv。')
            with ExitStack() as files, tempfile.TemporaryDirectory(dir=WORK_DIR) as tmp:
                fields=[]
                for c,p in zip(comps,paths):
                    ds=files.enter_context(xr.open_dataset(p,cache=False))
                    raw=canonical_raw_level(ds,c,level,p)
                    if not (np.array_equal(raw.latitude,latitude) and np.array_equal(raw.longitude,longitude)):
                        raise ValueError(f'{p}: 原始裁剪网格与风场网格不同。')
                    times=raw.time.values
                    trainval=int((1-.2)*(len(times)-1))
                    k=int((1-.1/(1-.2))*trainval)
                    reconstructed=times[6:6+k]
                    if not np.array_equal(reconstructed,expected):
                        raise ValueError(f'{p}: 从风场全集重建的 1 h 训练时间与 Text S7 不同。')
                    fields.append(raw.sel(time=slice(expected[0],expected[-1])))
                capacity=ntrain*len(latitude)*len(longitude)
                if shutil.disk_usage(tmp).free < capacity*8+256*1024**2:
                    raise OSError(f'{WORK_DIR} 空间不足，精确分位数工作文件需要 {capacity*8/2**30:.2f} GiB。')
                mm=np.memmap(Path(tmp)/'train.dat',dtype='f8',mode='w+',shape=(capacity,))
                n=0
                try:
                    for i in range(0,ntrain,BLOCK_SIZE*4):
                        key=slice(i,min(i+BLOCK_SIZE*4,ntrain))
                        a=fields[0].isel(time=key).values.astype('f8')
                        magnitude=(np.hypot(a,fields[1].isel(time=key).values.astype('f8'))
                                   if wi==0 else np.abs(a))
                        finite=magnitude[np.isfinite(magnitude)]
                        mm[n:n+finite.size]=finite
                        n+=finite.size
                    if n<3: raise ValueError(f'{wind}/{level}: 训练样本不足。')
                    cut=np.asarray(QUANTILES)*(n-1)
                    lo=np.floor(cut).astype(int); hi=np.ceil(cut).astype(int)
                    mm[:n].partition(np.unique(np.r_[lo,hi]))
                    vals=mm[lo]+(cut-lo)*(mm[hi]-mm[lo])
                finally:
                    del mm
            for q,t in zip(QUANTILES,vals):
                rows.append(dict(wind=wind,level_hpa=level,quantile=q,threshold=float(t),
                    units='m/s' if wi==0 else 'Pa/s',reference_split='train',
                    reference_start=start,reference_end=stop,training_time_count=ntrain,
                    n_training_finite=n,source=';'.join(map(str,paths))))
            print(wind, int(level), vals,flush=True)
    table=pd.DataFrame(rows)
    values=validate_train_table(table)
    path=WORK_DIR/'reconstructed_training_extreme_thresholds.csv'
    tmp=path.with_suffix('.tmp.csv')
    table.to_csv(tmp,index=False,float_format='%.17g');os.replace(tmp,path)
    return values,str(path)


In [ ]:
records,common_times,latitude,longitude,source_audit=scan_inputs()
ridge_audit=audit_ridge_headers()
if TEST_SUPPORT is not None:
    start,end,count=TEST_SUPPORT
    if (len(common_times)!=count or common_times[0]!=np.datetime64(start) or common_times[-1]!=np.datetime64(end)):
        raise ValueError(f'共同测试时次与 Fig. 7 不同：{len(common_times)}，{common_times[0]} 至 {common_times[-1]}。')
if EXPECTED_GRID is not None:
    for coord, spec in zip((latitude,longitude),EXPECTED_GRID):
        if not np.array_equal(coord,np.linspace(*spec)):
            raise ValueError('输入网格与正文评价网格不同。')
thresholds,threshold_source=obtain_thresholds()
settings=dict(protocol='S21-v1.0',train=TRAIN_SUPPORT,leads=LEADS,levels=LEVELS.tolist(),quantiles=QUANTILES,
    ridge_alpha=RIDGE_ALPHA,omega_factors=W_TO_OMEGA,threshold_values=thresholds.tolist(),threshold_source=threshold_source,
    common_count=len(common_times),common_start=str(common_times[0]),common_end=str(common_times[-1]),
    time_sha256=hashlib.sha256(common_times.astype('<i8').tobytes()).hexdigest(),
    sources=[(key,r['variable'],r['stat']) for key,r in records.items()])
signature=hashlib.sha256(json.dumps(settings,sort_keys=True,ensure_ascii=False).encode()).hexdigest()


In [ ]:
def block_center(values, grid, nblock, active, first):
    """Center within the block relative to its first observation; constant series stay exactly constant."""
    anchor=np.zeros(len(nblock))
    anchor[active]=values[first]
    shifted=values-anchor[grid]
    mean=np.divide(np.bincount(grid,weights=shifted,minlength=len(nblock)),nblock,
                   out=np.zeros_like(anchor),where=nblock>0)
    return anchor+mean,shifted-mean[grid]


def process_wind(wi):
    wind=WINDS[wi]
    grid_shape=(len(LEVELS),len(latitude),len(longitude))
    ng=int(np.prod(grid_shape))
    nh,nq,nm=len(LEADS),len(QUANTILES),len(GROUPS)
    per_grid=(nh,nq,ng)
    n=np.zeros(per_grid,dtype='i8')
    mx,vx=(np.zeros(per_grid) for _ in range(2))
    my,vy,cv,sse=(np.zeros((nm,)+per_grid) for _ in range(4))
    counts=np.zeros((nm,nh,nq,len(LEVELS),4),dtype='i8')
    comps=('u','v') if wi==0 else ('w',)
    with ExitStack() as stack:
        open_ds={}
        fields={}
        for (group,h,component,kind),r in records.items():
            if component not in comps: continue
            path=r['path']
            if path not in open_ds:
                open_ds[path]=stack.enter_context(xr.open_dataset(path,cache=False))
            field_key=(path,r['variable'])
            if field_key not in fields:
                fields[field_key]=canonical_field(open_ds[path],r['variable'],path)

        def read(group,h,c,kind,sl):
            row=records[group,h,c,kind]
            fld=fields[row['path'],row['variable']]
            idx=row['indices'][sl]
            z=fld.isel(time=time_selection(idx)).values.astype(np.float64,copy=False)
            return z*W_TO_OMEGA[group] if c=='w' and W_TO_OMEGA[group]!=1 else z

        for begin in range(0,len(common_times),BLOCK_SIZE):
            end=min(begin+BLOCK_SIZE,len(common_times)); sl=slice(begin,end)
            series={}
            valid=np.ones((end-begin,)+grid_shape,dtype=bool)
            for h in LEADS:
                ref=[]
                forecasts={g:[] for g in GROUPS}
                for c in comps:
                    truth=read('B',h,c,'testy',sl)
                    exact_truth_check(truth,read('Ridge',h,c,'testy',sl),f'{wind}, {h}h, {begin}:{end}')
                    ref.append(truth)
                    for g in GROUPS:
                        forecasts[g].append(read(g,h,c,'predicty',sl))
                convert=lambda xs: np.hypot(*xs) if wi==0 else xs[0]
                obs=convert(ref)
                preds=[convert(forecasts[g]) for g in GROUPS]
                valid &= np.isfinite(obs)
                for pred in preds: valid &= np.isfinite(pred)
                series[h]=(obs,preds)
            valid_per_level=valid.sum(axis=(0,2,3))
            for j,h in enumerate(LEADS):
                obs,preds=series[h]
                obs_intensity=obs if wi==0 else np.abs(obs)
                for k,q in enumerate(QUANTILES):
                    cut=thresholds[wi,:,k][None,:,None,None]
                    event=valid & (obs_intensity>=cut)
                    nobs=event.sum(axis=(0,2,3))
                    time_i,grid=np.nonzero(event.reshape(end-begin,ng))
                    bn=np.bincount(grid,minlength=ng)
                    active,first=np.unique(grid,return_index=True)
                    truth_values=obs.reshape(end-begin,ng)[time_i,grid]
                    bx,dx=block_center(truth_values,grid,bn,active,first)
                    total=n[j,k]+bn
                    fraction=np.divide(bn,total,out=np.zeros(ng),where=total>0)
                    factor=n[j,k]*fraction
                    shift_x=bx-mx[j,k]
                    for m,pred in enumerate(preds):
                        forecast_event=valid & ((pred if wi==0 else np.abs(pred))>=cut)
                        nf=forecast_event.sum(axis=(0,2,3))
                        hits=(event & forecast_event).sum(axis=(0,2,3))
                        counts[m,j,k]+=np.stack([hits,nf-hits,nobs-hits,valid_per_level-nobs-nf+hits],axis=-1)
                        pvalues=pred.reshape(end-begin,ng)[time_i,grid]
                        by,dy=block_center(pvalues,grid,bn,active,first)
                        shift_y=by-my[m,j,k]
                        my[m,j,k]+=shift_y*fraction
                        vy[m,j,k]+=np.bincount(grid,weights=dy*dy,minlength=ng)+shift_y**2*factor
                        cv[m,j,k]+=np.bincount(grid,weights=dx*dy,minlength=ng)+shift_x*shift_y*factor
                        sse[m,j,k]+=np.bincount(grid,weights=(pvalues-truth_values)**2,minlength=ng)
                    mx[j,k]+=shift_x*fraction
                    vx[j,k]+=np.bincount(grid,weights=dx*dx,minlength=ng)+shift_x**2*factor
                    n[j,k]=total
            if begin==0 or end==len(common_times) or (begin//BLOCK_SIZE+1)%100==0:
                print(f'{wind}: {end:,}/{len(common_times):,} target times',flush=True)
    if np.any(counts<0): raise ArithmeticError('负的列联计数。')
    H,F,M,C=(counts[...,i] for i in range(4)); N=counts.sum(axis=-1)
    if not np.array_equal(H+M,np.broadcast_to((H+M)[:1],H.shape)):
        raise ArithmeticError('模型的观测极端事件数不同。')
    if not np.array_equal(N,np.broadcast_to(N[:1,:1,:1,:],N.shape)):
        raise ArithmeticError('三个模型、六个时效、三个阈值未共享有效样本。')
    random=np.divide((H+F).astype('f8')*(H+M),N,out=np.full(H.shape,np.nan),where=N>0)
    den=H+F+M-random
    ets=np.divide(H-random,den,out=np.full(H.shape,np.nan),where=den>0)
    if np.any(np.isfinite(ets)&((ets>1+1e-10)|(ets < -1/3-1e-10))):
        raise ArithmeticError('ETS超出理论范围。')
    enough=n>=3
    denom=np.sqrt(vx[None])*np.sqrt(vy)
    corr=np.divide(cv,denom,out=np.full_like(cv,np.nan),where=enough[None]&(denom>0))
    if np.any(np.isfinite(corr)&(np.abs(corr)>1+1e-9)): raise ArithmeticError('R超出舍入误差范围。')
    corr=np.clip(corr,-1,1)
    rmse=np.sqrt(np.divide(sse,n[None],out=np.full_like(sse,np.nan),where=enough[None]))
    dims=('model','lead_hour','percentile','level')
    out=xr.Dataset(coords=dict(model=list(MODEL_NAMES),lead_hour=list(LEADS),
        percentile=[int(100*q) for q in QUANTILES],level=LEVELS))
    for name,arr in [('hits',H),('false_alarms',F),('misses',M),('correct_negatives',C),
                     ('valid_count',N),('expected_hits',random),('ETS',ets)]:
        out[name]=(dims,arr)
    for name,arr in [('R',corr),('RMSE',rmse)]:
        maps=arr.reshape(nm,nh,nq,*grid_shape)
        shared=np.all(np.isfinite(maps),axis=(0,1,2))
        ngrid=shared.sum(axis=(-2,-1))
        totals=np.where(shared[None,None,None],maps,0.).sum(axis=(-2,-1))
        out[name]=(dims,np.divide(totals,ngrid,out=np.full_like(totals,np.nan),where=ngrid>0))
        out[name+'_grid_count']=('level',ngrid)
    out['training_threshold']=(('percentile','level'),thresholds[wi].T)
    return out.expand_dims(quantity=[wind])


def results_for_all_models():
    assert_sources_unchanged()
    parts=[]
    for wi,wind in enumerate(WINDS):
        cache=WORK_DIR/f'completed_{wind}.nc'
        use_cache=False
        if cache.is_file():
            try:
                with xr.open_dataset(cache) as previous:
                    if previous.attrs.get('input_signature')==signature:
                        part=previous.load();use_cache=True
            except (OSError,ValueError) as exc:
                warnings.warn(f'无法读取缓存，将重新计算 {wind}: {exc}')
        if not use_cache:
            part=process_wind(wi)
            part.attrs.update(input_signature=signature)
            assert_sources_unchanged()
            write_nc(part,cache)
        else: print(f'{wind}: 复用已完成的同输入缓存。')
        parts.append(part)
    ds=xr.concat(parts,dim='quantity')
    for name in ('ETS','R','RMSE'):
        ds[name+'_gain']=xr.concat([
            (ds[name].isel(model=i,drop=True)-ds[name].isel(model=j,drop=True))*(1 if name!='RMSE' else -1)
            for i,j in PAIR_INDEX],dim=pd.Index(PAIR_NAMES,name='comparison'))
    ds=ds.assign_coords(test_valid_time=('test_valid_time',common_times))
    ds.level.attrs['units']='hPa'
    for name in ('RMSE','RMSE_gain','training_threshold'):
        ds[name].attrs['units_by_quantity']='horizontal: m s-1; vertical: Pa s-1'
    ds.attrs.update(input_signature=signature,ridge_alpha=RIDGE_ALPHA,
        threshold_source=threshold_source,evaluation_start=str(common_times[0]),
        evaluation_end=str(common_times[-1]),evaluation_hours=len(common_times),
        conditional='Pearson R and RMSE on observed extreme times, >=3 per grid; signed omega',
        event_rule='forecast/reference S or |omega| >= levelwise training threshold; all paired-valid samples',
        support='Three methods and six leads share finite grid-times per quantity and level; '
                'continuous spatial masks shared across all methods, thresholds and leads per metric and level',
        plot_aggregation='Equal-weight mean of three levelwise scores (all three required)',
        gain='First minus second for ETS/R; second minus first for RMSE; positive favours first')
    return ds


In [ ]:
scores=results_for_all_models()
write_nc(scores,WORK_DIR/'FigS21_scores.nc')
